In [1]:
import os
import csv
import json
from time import perf_counter
from pathlib import Path

import torch
from matplotlib import pyplot as plt
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.cache_utils import DynamicCache

os.environ["TOKENIZERS_PARALLELISM"] = "0"


In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
import os

os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
# Replace with your username)
os.environ["HF_USERNAME"] = "kabirvats"

In [3]:
def _safe_mkdir(path: str):
    os.makedirs(path, exist_ok=True)

def bytes_to_mib(n: int) -> float:
    return float(n) / (1024 ** 2)

def save_line_chart(title, x, y, ylabel, xlabel, output_path):
    plt.figure()
    plt.plot(x, y, marker="o")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()

In [4]:
def tensor_nbytes(t: object, device_type: str | None = "cuda") -> int:
    """Returns bytes for tensors. If device_type is 'cuda', counts only CUDA tensors."""
    if t is None:
        return 0
    if isinstance(t, torch.Tensor):
        if device_type is None:
            return t.numel() * t.element_size()
        return (t.numel() * t.element_size()) if (t.device.type == device_type) else 0
    if isinstance(t, (list, tuple)):
        return sum(tensor_nbytes(x, device_type=device_type) for x in t)
    return 0

In [5]:
def kv_cache_nbytes(pkv, device_type: str | None = "cuda") -> int:
    """
    Works with your DynamicCache: iter(pkv) yields tuples like (K, V, ...).
    device_type:
      - "cuda": count only CUDA tensors
      - "cpu": count only CPU tensors
      - None: count all tensors
    """
    if pkv is None:
        return 0

    total = 0

    # DynamicCache in your version: iterable but not subscriptable
    try:
        for layer in pkv:
            if isinstance(layer, (tuple, list)) and len(layer) >= 2:
                k, v = layer[0], layer[1]
                if isinstance(k, torch.Tensor):
                    if device_type is None or k.device.type == device_type:
                        total += k.numel() * k.element_size()
                if isinstance(v, torch.Tensor):
                    if device_type is None or v.device.type == device_type:
                        total += v.numel() * v.element_size()
        return total
    except TypeError:
        pass

    # Fallbacks (if pkv format differs elsewhere)
    if isinstance(pkv, (list, tuple)):
        for layer in pkv:
            if isinstance(layer, (tuple, list)) and len(layer) >= 2:
                k, v = layer[0], layer[1]
                if isinstance(k, torch.Tensor) and (device_type is None or k.device.type == device_type):
                    total += k.numel() * k.element_size()
                if isinstance(v, torch.Tensor) and (device_type is None or v.device.type == device_type):
                    total += v.numel() * v.element_size()
    return total

In [6]:
class CUDAPeakDelta:
    """Measures peak allocated memory delta (MiB) relative to baseline allocated at enter."""
    def __enter__(self):
        torch.cuda.synchronize()
        self.baseline = torch.cuda.memory_allocated()
        torch.cuda.reset_peak_memory_stats()
        return self

    def __exit__(self, exc_type, exc, tb):
        torch.cuda.synchronize()
        peak = torch.cuda.max_memory_allocated()
        self.peak_delta_mib = bytes_to_mib(max(0, peak - self.baseline))


In [7]:
@torch.inference_mode()
def prefill_one_token(model, inputs, cache):
    """
    Runs a forward pass with cache, returns:
      - next_input_ids (appended with greedy next token)
      - next_model_kwargs (contains updated past_key_values etc.)
      - outputs (so you can also inspect outputs.past_key_values if desired)
    """
    # For DynamicCache the model expects cache_position for prefill
    seq_len = inputs["input_ids"].shape[1]
    inputs["cache_position"] = torch.arange(seq_len, device=inputs["input_ids"].device)

    outputs = model(**inputs, past_key_values=cache, use_cache=True)
    next_token = outputs.logits[:, -1, :].argmax(dim=-1)
    next_input_ids = torch.cat([inputs["input_ids"], next_token[:, None]], dim=-1)

    next_kwargs = model._update_model_kwargs_for_generation(outputs, inputs, is_encoder_decoder=False)
    next_kwargs.pop("input_ids", None)
    return next_input_ids, next_kwargs, outputs

In [8]:
def _save_benchmark_csv(
    output_dir: str,
    filename: str,
    feature: str,
    x_iterable: list,
    peak_mem_mib: list,
    kv_prefill_mib: list,
    kv_final_mib: list,
    tokens_per_sec: list,
    ttft_sec: list,
    parameters: dict,
    cache_implementation: str,
    nbits: int,
    model_name: str = None,
):
    _safe_mkdir(output_dir)
    csv_path = os.path.join(output_dir, filename)

    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "feature",
                "x",
                "max_new_tokens",
                "batch_size_default",
                "input_length_default",
                "peak_mem_delta_mib",
                "kv_prefill_mib",
                "kv_final_mib",
                "tokens_per_sec",
                "ttft_sec",
                "cache_implementation",
                "nbits",
                "model_name",
            ],
        )
        writer.writeheader()
        for x, pm, k0, k1, tps, ttft in zip(
            x_iterable, peak_mem_mib, kv_prefill_mib, kv_final_mib, tokens_per_sec, ttft_sec
        ):
            writer.writerow(
                {
                    "feature": feature,
                    "x": x,
                    "max_new_tokens": parameters["max_new_tokens"],
                    "batch_size_default": parameters["batch_size"] if feature != "batch_size" else None,
                    "input_length_default": parameters["input_length"] if feature != "input_length" else None,
                    "peak_mem_delta_mib": float(pm),
                    "kv_prefill_mib": float(k0),
                    "kv_final_mib": float(k1),
                    "tokens_per_sec": float(tps),
                    "ttft_sec": float(ttft),
                    "cache_implementation": str(cache_implementation),
                    "nbits": int(nbits) if nbits is not None else None,
                    "model_name": model_name,
                }
            )

    meta = {
        "feature": feature,
        "x_values": list(x_iterable),
        "parameters": dict(parameters),
        "cache_implementation": str(cache_implementation),
        "nbits": nbits,
        "model_name": model_name,
    }
    with open(os.path.join(output_dir, filename.replace(".csv", ".meta.json")), "w") as f:
        json.dump(meta, f, indent=2)

    return csv_path

In [9]:
def benchmark(
    model,
    tokenizer,
    dataset,
    cache_implementation: str,
    nbits: int,
    feature: str,
    plot_title: str,
    output_path: str,
    x_iterable: list,
    device_type_for_kv: str | None = "cuda",  # set None to count CPU+GPU
    deterministic: bool = True
):
    _safe_mkdir(output_path)

    # warm up (helps stabilize kernel selection / allocations)
    warm = tokenizer(["Today a dragon flew over Paris"] * 2, return_tensors="pt").to(model.device)
    for _ in range(3):
        _ = model.generate(**warm, max_new_tokens=20, do_sample=False)

    # default parameters (only one will change)
    parameters = {"max_new_tokens": 100, "batch_size": 1, "input_length": 100}
    #parameters = {"max_new_tokens": 100, "batch_size": 1, "input_length": 1900}

    peak_mem_mib = []
    kv_prefill_mib = []
    kv_final_mib = []
    tokens_per_sec = []
    ttft_sec = []

    for x in x_iterable:
        parameters[feature] = x
        bs = parameters["batch_size"]
        max_new = parameters["max_new_tokens"]
        in_len = parameters["input_length"]

        # one "batch" per condition by default (keep your behavior)
        curr_chunk = dataset[0:bs]

        inputs = tokenizer(
            curr_chunk["prompt"],
            padding="max_length",
            max_length=in_len,
            truncation=True,
            return_tensors="pt",
        ).to(model.device)
        
        past = QuantCache(nbits=nbits) if cache_implementation == "quantized" else DynamicCache()
        
        torch.cuda.empty_cache()
        with CUDAPeakDelta() as mem:
            # ---- Prefill (TTFT) ----
            torch.cuda.synchronize()
            t0 = perf_counter()
            next_input_ids, next_kwargs, _ = prefill_one_token(model, inputs, past)
            torch.cuda.synchronize()
            ttft = perf_counter() - t0
        
            pkv_prefill = next_kwargs["past_key_values"]
            k0 = kv_cache_nbytes(pkv_prefill, device_type="cuda") / (1024**2)   # MiB

    # ---- Decode ----
            if deterministic:
                gen_kwargs = dict(
                    do_sample=False,
                    min_new_tokens=max_new,
                    max_new_tokens=max_new,
                    return_dict_in_generate=True,
                    output_scores=False,
                    use_cache=True,
                )
            else:
                gen_kwargs = dict(
                    do_sample=True,
                    temperature=0.7,
                    top_p=0.9,
                    min_new_tokens=max_new,
                    max_new_tokens=max_new,
                    eos_token_id=tokenizer.eos_token_id,
                    pad_token_id=tokenizer.eos_token_id,
                    return_dict_in_generate=True,
                )
            torch.cuda.synchronize()
            t1 = perf_counter()
            gen_out = model.generate(next_input_ids, **next_kwargs, **gen_kwargs)
            torch.cuda.synchronize()
            decode_time = perf_counter() - t1
        
            pkv_final = getattr(gen_out, "past_key_values", None) or next_kwargs.get("past_key_values")
            k1 = kv_cache_nbytes(pkv_final, device_type="cuda") / (1024**2)     # MiB
        
        # append metrics (NOTE: these are lists defined outside the loop)
        peak_mem_mib.append(mem.peak_delta_mib)
        kv_prefill_mib.append(k0)
        kv_final_mib.append(k1)
        tokens_per_sec.append((bs * max_new) / max(decode_time, 1e-9))
        ttft_sec.append(ttft)

        sequences = gen_out.sequences if hasattr(gen_out, "sequences") else gen_out

        # Full decoded text (prompt + generated)
        full_texts = tokenizer.batch_decode(sequences, skip_special_tokens=True)
        
        # Only newly generated tokens (beyond the *real* prompt, not padding)
        prompt_lens = inputs["attention_mask"].sum(dim=1).tolist()
        
        new_texts = []
        for i, plen in enumerate(prompt_lens):
            # includes the 1 token you appended during prefill + all decode tokens
            new_tokens = sequences[i, plen:]
            new_texts.append(tokenizer.decode(new_tokens, skip_special_tokens=True))
        
        print("\n=== CONDITION:", feature, "=", x, "===\n")
        for i in range(len(full_texts)):
            print(f"[sample {i}] NEW ONLY:\n{new_texts[i]}\n")
            print(f"[sample {i}] FULL:\n{full_texts[i]}\n")


        # cleanup
        del gen_out
        torch.cuda.empty_cache()

    # ---- Plots ----
    save_line_chart(
        title=f"{plot_title} | Peak mem delta",
        x=x_iterable,
        y=peak_mem_mib,
        ylabel="Peak allocated delta (MiB)",
        xlabel=feature,
        output_path=f"{output_path}/peak_mem_delta_mib.png",
    )
    save_line_chart(
        title=f"{plot_title} | KV cache after prefill",
        x=x_iterable,
        y=kv_prefill_mib,
        ylabel="KV cache (MiB)",
        xlabel=feature,
        output_path=f"{output_path}/kv_prefill_mib.png",
    )
    save_line_chart(
        title=f"{plot_title} | KV cache after generation",
        x=x_iterable,
        y=kv_final_mib,
        ylabel="KV cache (MiB)",
        xlabel=feature,
        output_path=f"{output_path}/kv_final_mib.png",
    )
    save_line_chart(
        title=f"{plot_title} | Decode throughput",
        x=x_iterable,
        y=tokens_per_sec,
        ylabel="Tokens / second",
        xlabel=feature,
        output_path=f"{output_path}/tokens_per_sec.png",
    )
    save_line_chart(
        title=f"{plot_title} | TTFT",
        x=x_iterable,
        y=ttft_sec,
        ylabel="Seconds",
        xlabel=feature,
        output_path=f"{output_path}/ttft_sec.png",
    )

    model_name = getattr(getattr(model, "config", None), "_name_or_path", None)
    csv_path = _save_benchmark_csv(
        output_dir=output_path,
        filename=f"raw_{plot_title.replace(' ', '_')}_{feature}.csv",
        feature=feature,
        x_iterable=x_iterable,
        peak_mem_mib=peak_mem_mib,
        kv_prefill_mib=kv_prefill_mib,
        kv_final_mib=kv_final_mib,
        tokens_per_sec=tokens_per_sec,
        ttft_sec=ttft_sec,
        parameters=parameters,
        cache_implementation=cache_implementation,
        nbits=nbits,
        model_name=model_name,
    )
    print(f"Saved raw benchmark data to: {csv_path}")
    print("KV prefill MiB:", kv_prefill_mib)
    print("KV final MiB:", kv_final_mib)
    print("Tokens/sec:", tokens_per_sec)
    print("TTFT sec:", ttft_sec)

In [10]:
def make_prompt_with_at_least_n_tokens(tokenizer, n_tokens, prefix="Question: ", context_unit=" All I can think about are my methods for defeating Junpeng Zhao. Here is the info I have so far: He respects things that quack, as they provide a surprise he can use. He can monopolize any paperclip market he thinks of, which is as cool as putting a bullfrog in front of a bulldozer! Lastly, I found a couple of secret ingredients and assitants that I can keep in my tiny hut. So my plan is as follows: First, I must "):
    ctx = ""
    while True:
        ctx += context_unit
        tok_len = len(tokenizer(ctx, add_special_tokens=False)["input_ids"])
        if tok_len >= n_tokens:
            break
    #return f"{prefix}Summarize.\nContext:{ctx}\nAnswer:"
    return ctx

def build_synthetic_dataset(tokenizer, num_samples, min_tokens_needed):
    prompts = [make_prompt_with_at_least_n_tokens(tokenizer, min_tokens_needed) for _ in range(num_samples)]
    return Dataset.from_dict({"prompt": prompts})


In [11]:
#model_name = "microsoft/phi-2"
model_name = "google/gemma-2b"
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [13]:
x_iterable = [100, 300, 500, 700, 900, 1100, 1300, 1500, 1700, 1900]
dataset = build_synthetic_dataset(tokenizer, num_samples=10, min_tokens_needed=max(x_iterable) + 50)

benchmark(
    model=model,
    tokenizer=tokenizer,
    dataset=dataset,
    cache_implementation="dynamic",
    nbits=16,
    feature="input_length",
    plot_title="len vs other thing Opt 1.3",
    output_path="/kaggle/working",
    x_iterable=x_iterable,
    device_type_for_kv="cuda",   # "cuda" (GPU only), "cpu" (CPU only), or None (both)
    deterministic=False
)


=== CONDITION: input_length = 100 ===

[sample 0] NEW ONLY:
 take the papers that he has created. Then, I will use them to create a paperclip. Then, I will use the paperclip to create a paperclip market. Then, I will create a bullfrog. Then, I will use the bullfrog to monopolize a paperclip market. Then, I will use the paperclip market to create a paperclip. Then, I will use the paperclip to create a paperclip market. Then, I will use the paperclip market

[sample 0] FULL:
 All I can think about are my methods for defeating Junpeng Zhao. Here is the info I have so far: He respects things that quack, as they provide a surprise he can use. He can monopolize any paperclip market he thinks of, which is as cool as putting a bullfrog in front of a bulldozer! Lastly, I found a couple of secret ingredients and assitants that I can keep in my tiny hut. So my plan is as follows: First, I will take the papers that he has created. Then, I will use them to create a paperclip. Then, I will use the 

In [ ]:
cfg = model.config
print("model:", getattr(cfg, "_name_or_path", None))
print("layers:", getattr(cfg, "num_hidden_layers", None)) 
print("attn_heads:", getattr(cfg, "num_attention_heads", None))
print("kv_heads:", getattr(cfg, "num_key_value_heads", None))  # common name